# 03 - Model training

**Goal:** predict `drop_outcome` (GPH falls by 10+ points on chemotherapy) from information available before the first dose.

All logic lives in `src/` so the notebook, the command line and the Streamlit app use the same code:

| File | What it does |
|---|---|
| `src/config.py` | paths, column lists, leakage columns, seed |
| `src/data.py` | Person 1's stratified 80/20 split + preprocessing (rebuilt inside every model pipeline) |
| `src/models.py` | the five models and their hyperparameter grids |
| `src/train.py` | grid search with 5-fold CV on AUROC, voting ensemble, final-model selection |

Plan followed: (1) Logistic Regression baseline, (2) KNN, SVM, Random Forest, Gradient Boosting, (5) soft-voting ensemble.

In [1]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")
import pandas as pd
pd.set_option("display.precision", 3)
from src import config, data
from src.models import model_specs

## 1. Data handed over by Person 1
Same split as `data/processed/` (checked below): identical patients, identical transformed values.

In [2]:
X_train, X_test, y_train, y_test = data.split()
print("train", X_train.shape, "positives", f"{y_train.mean():.1%}")
print("test ", X_test.shape, "positives", f"{y_test.mean():.1%}")

import numpy as np
pre = data.build_preprocessor().fit(X_train)
ref = pd.read_csv(ROOT / "data/processed/X_test.csv")
print("matches Person 1's X_test.csv:",
      list(pre.get_feature_names_out()) == list(ref.columns) and np.allclose(pre.transform(X_test), ref.values))
print("leakage columns excluded:", config.LEAKAGE_COLS)

train (1440, 52) positives 20.4%
test  (360, 52) positives 20.3%
matches Person 1's X_test.csv: True
leakage columns excluded: ['patient_id', 'on_chemo_gph', 'drop_outcome', 'threshold_outcome']


## 2. Why preprocessing sits inside each pipeline

Each model is `Pipeline([preprocess, classifier])`. During cross-validation the imputer medians and scaler means are refit on the 4 training folds only, so the validation fold never leaks into them. It also means the saved model accepts raw patient values, which is what the demo app sends.

## 3. Models and the settings we tune

In [3]:
pd.DataFrame([{"Model": n, "Classifier": type(e).__name__,
               "Grid": {k.replace('clf__', '').replace('estimator__', ''): list(v) for k, v in g.items()},
               "Combinations": int(np.prod([len(v) for v in g.values()]))}
              for n, (e, g) in model_specs().items()])

,Model,Classifier,Grid,Combinations
0,Logistic Regression,LogisticRegression,"{'C': [0.001, 0.01, 0.1, 1.0, 10.0, 100.0], 'l...",18
1,KNN,KNeighborsClassifier,"{'n_neighbors': [5, 11, 21, 41, 71, 101, 151, ...",16
2,SVM,CalibratedClassifierCV,"{'C': [0.01, 0.1, 1, 10], 'kernel': ['linear',...",8
3,Random Forest,RandomForestClassifier,"{'min_samples_leaf': [1, 5, 20], 'max_features...",6
4,Gradient Boosting,GradientBoostingClassifier,"{'learning_rate': [0.03, 0.1], 'n_estimators':...",8


Notes for Q&A:
* **Logistic Regression (baseline):** a weighted sum of features turned into a probability. `C` = inverse regularisation strength; `l1_ratio` 0 = Ridge, 1 = LASSO (drops useless features), 0.5 = Elastic Net. Covers the paper's three logistic models.
* **KNN:** predicts from the k most similar training patients. Needs scaled features.
* **SVM:** finds the widest-margin boundary between classes; Platt scaling turns its scores into probabilities.
* **Random Forest:** many decision trees on random subsets of rows/features, averaged.
* **Gradient Boosting:** trees built one after another, each fixing the previous ones' errors.
* **Class imbalance (20% positive):** handled by choosing the decision threshold that maximises F1 on out-of-fold training predictions, instead of using 0.5. Probabilities stay interpretable as real risks.

## 4. Train everything (about 2 minutes)

In [4]:
from src.train import main as train
results, final = train("drop_outcome")

Target: drop_outcome | train (1440, 52), positives 20.4%


  Logistic Regression  CV AUROC 0.7534  {'C': 0.1, 'l1_ratio': 1.0}  (29s)


  KNN                  CV AUROC 0.7193  {'n_neighbors': 201, 'weights': 'uniform'}  (2s)


  SVM                  CV AUROC 0.7331  {'C': 0.1, 'kernel': 'linear'}  (26s)


  Random Forest        CV AUROC 0.7409  {'max_features': 0.3, 'min_samples_leaf': 20}  (35s)


  Gradient Boosting    CV AUROC 0.7384  {'learning_rate': 0.03, 'max_depth': 2, 'n_estimators': 100}  (21s)


  Voting Ensemble      CV AUROC 0.7520  members=['Logistic Regression', 'Random Forest', 'Gradient Boosting']
Final model: Logistic Regression


In [5]:
summary = pd.DataFrame(results).T[["cv_auroc", "cv_auroc_std", "threshold", "best_params"]]
summary.sort_values("cv_auroc", ascending=False)

,cv_auroc,cv_auroc_std,threshold,best_params
Logistic Regression,0.753,0.061,0.21,"{'C': 0.1, 'l1_ratio': 1.0}"
Voting Ensemble,0.752,0.056,0.2,"{'members': ['Logistic Regression', 'Random Fo..."
Random Forest,0.741,0.051,0.23,"{'max_features': 0.3, 'min_samples_leaf': 20}"
Gradient Boosting,0.738,0.052,0.18,"{'learning_rate': 0.03, 'max_depth': 2, 'n_est..."
SVM,0.733,0.041,0.22,"{'C': 0.1, 'kernel': 'linear'}"
KNN,0.719,0.039,0.18,"{'n_neighbors': 201, 'weights': 'uniform'}"


## 5. Choosing the final model

The final model is the one with the best **cross-validated AUROC on the training set**. Ties within 0.002 go to the simpler model. The test set is not used for this choice, so the test score in notebook 04 is an honest estimate.

Saved to `models/final_model.joblib` together with its threshold and column lists. The paper's second outcome can be trained with `python -m src.train --target threshold_outcome`.

In [6]:
import joblib
b = joblib.load(ROOT / "models/final_model.joblib")
print(b["model_name"], "| CV AUROC", round(b["cv_auroc"], 4), "| threshold", b["threshold"])
lr = b["model"].named_steps["clf"] if hasattr(b["model"], "named_steps") else None
if lr is not None and hasattr(lr, "coef_"):
    names = b["model"].named_steps["preprocess"].get_feature_names_out()
    coefs = pd.Series(lr.coef_[0], index=names)
    print(f"non-zero coefficients: {(coefs != 0).sum()} of {len(coefs)}")
    display(coefs[coefs != 0].sort_values(key=abs, ascending=False).to_frame("coefficient (per SD)"))

Logistic Regression | CV AUROC 0.7534 | threshold 0.21
non-zero coefficients: 35 of 89


,coefficient (per SD)
num__stage,0.397
num__hemoglobin_g_dl,-0.379
num__albumin_g_dl,-0.315
num__baseline_gmh,-0.309
num__ecog_status,0.272
num__baseline_gph,0.257
num__num_comorbidities,0.193
num__spo2,-0.165
num__weight_loss_pct_6m,0.119
num__age,0.108
